In [ ]:
images_path = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-small/images/"
styles_path = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-small/styles.csv"
# images_path = "/kaggle/input/images/"
# styles_path = "/kaggle/input/styles.csv"
dataset_path = "/kaggle/working"

In [ ]:
import pandas as pd
import os
import shutil
import matplotlib.pyplot as plt

# Read the styles.csv file
styles_df = pd.read_csv(styles_path, on_bad_lines='skip')

# Distribusi data sebelum cleaning (semua data)
if "season" not in styles_df.columns:
    print("Kolom 'season' tidak ditemukan. Distribusi sebelum cleaning dilewati.")
elif styles_df.empty:
    print("styles_df kosong. Distribusi sebelum cleaning dilewati.")
else:
    season_counts_all = (
        styles_df["season"]
        .fillna("Unknown")
        .astype(str)
        .value_counts()
    )
    print("Distribusi gambar berdasarkan musim (sebelum cleaning):")
    for season, count in season_counts_all.items():
        print(f"{season}: {count} gambar")

    plt.figure(figsize=(10, 6))
    plt.bar(
        season_counts_all.index.tolist(),
        season_counts_all.values.tolist(),
        color="skyblue",
    )
    plt.xlabel("Musim")
    plt.ylabel("Jumlah Gambar")
    plt.title("Distribusi Gambar Berdasarkan Musim (Sebelum Cleaning)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

styles_df = styles_df[styles_df['masterCategory'] == 'Apparel']

# Create the main dataset directory if it doesn't exist
os.makedirs(dataset_path, exist_ok=True)

# Iterate through each row in the dataframe
for index, row in styles_df.iterrows():
    image_id = row['id']
    season = row['season']

    # Handle potential NaN values in the 'season' column
    if pd.isna(season):
        season = 'Unknown' # or any other default string

    # Ensure season is treated as a string
    season = str(season)

    # Create a directory for the season if it doesn't exist
    season_path = os.path.join(dataset_path, season)
    os.makedirs(season_path, exist_ok=True)

    # Define the source and destination paths for the image
    source_image_path = os.path.join(images_path, str(image_id) + '.jpg')
    destination_image_path = os.path.join(season_path, str(image_id) + '.jpg')

    # Copy the image to the season directory
    if os.path.exists(source_image_path):
        shutil.copy(source_image_path, destination_image_path)
    else:
        print(f"Warning: Image not found for ID {image_id} at {source_image_path}")

print("Images sorted by season successfully!")

In [ ]:
!rm -r /kaggle/working/Unknown/

In [ ]:
import os

# Get the list of season directories
season_directories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

# Count images in each season directory
season_counts = {}
for season_dir in season_directories:
    season_path = os.path.join(dataset_path, season_dir)
    image_files = [f for f in os.listdir(season_path) if os.path.isfile(os.path.join(season_path, f))]
    season_counts[season_dir] = len(image_files)

# Print the distribution
print("Distribusi gambar berdasarkan musim:")
for season, count in season_counts.items():
    print(f"{season}: {count} gambar")

In [ ]:
import matplotlib.pyplot as plt

# Get the season names and counts from the dictionary
seasons = list(season_counts.keys())
counts = list(season_counts.values())

# Create a bar plot
plt.figure(figsize=(10, 6))
plt.bar(seasons, counts, color=['skyblue', 'lightgreen', 'salmon', 'gold'])
plt.xlabel("Musim")
plt.ylabel("Jumlah Gambar")
plt.title("Distribusi Gambar Berdasarkan Musim")
plt.show()

In [ ]:
import os
import random

sample_image_path = None
sample_image_season = None

if os.path.isdir(dataset_path):
    season_dirs = [
        d for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ]
    random.shuffle(season_dirs)

    for season in season_dirs:
        season_path = os.path.join(dataset_path, season)
        image_files = [
            f for f in os.listdir(season_path)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]
        if image_files:
            sample_image_file = random.choice(image_files)
            sample_image_path = os.path.join(season_path, sample_image_file)
            sample_image_season = season
            break

if sample_image_path is None:
    print("Sample image tidak ditemukan. Pastikan dataset_path berisi folder season dan gambar.")
else:
    print(f"Sample image: {sample_image_path}")
    print(f"Season: {sample_image_season}")

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image

# Get the list of season directories and their counts
season_directories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

# Recalculate season_counts in case the previous cell was not run
season_counts = {}
for season_dir in season_directories:
    season_path = os.path.join(dataset_path, season_dir)
    image_files = [f for f in os.listdir(season_path) if os.path.isfile(os.path.join(season_path, f))]
    season_counts[season_dir] = len(image_files)


# Determine the target number of images (e.g., the count of the season with the most images)
target_count = max(season_counts.values())

# Define data augmentation parameters
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Apply data augmentation to seasons with fewer images
for season_dir in season_directories:
    current_count = season_counts[season_dir]

    if current_count < target_count:
        season_path = os.path.join(dataset_path, season_dir)
        images_to_augment = [os.path.join(season_path, f) for f in os.listdir(season_path) if os.path.isfile(os.path.join(season_path, f))]

        # Calculate how many images to generate
        num_augment_needed = target_count - current_count
        if num_augment_needed <= 0:
            continue

        # Generate and save augmented images
        augmented_count = 0
        for img_path in images_to_augment:
            img = Image.open(img_path)
            x = tf.keras.preprocessing.image.img_to_array(img)
            x = x.reshape((1,) + x.shape)  # Reshape to (1, height, width, channels)

            i = 0
            for batch in datagen.flow(x, batch_size=1, save_to_dir=season_path, save_prefix='aug', save_format='jpeg'):
                i += 1
                augmented_count += 1
                if i >= num_augment_needed / len(images_to_augment) or augmented_count >= num_augment_needed:
                    break
            if augmented_count >= num_augment_needed:
                break


print("Data augmentation applied to balance the dataset.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

if sample_image_path is None:
    print("Sample image belum tersedia. Jalankan cell pemilihan sample image.")
elif "datagen" not in globals():
    print("ImageDataGenerator belum dibuat. Jalankan cell augmentasi terlebih dahulu.")
else:
    vis_img_height = globals().get("IMG_HEIGHT", 80)
    vis_img_width = globals().get("IMG_WIDTH", 60)

    img = tf.keras.utils.load_img(
        sample_image_path,
        target_size=(vis_img_height, vis_img_width)
    )
    img_array = tf.keras.utils.img_to_array(img)
    img_batch = np.expand_dims(img_array, axis=0)

    plt.figure(figsize=(12, 3))
    plt.subplot(1, 5, 1)
    plt.imshow(img)
    plt.title("Sebelum")
    plt.axis("off")

    aug_iter = datagen.flow(img_batch, batch_size=1)
    for i in range(4):
        aug_img = next(aug_iter)[0].astype("uint8")
        plt.subplot(1, 5, i + 2)
        plt.imshow(aug_img)
        plt.title(f"Aug {i + 1}")
        plt.axis("off")

    if sample_image_season is not None:
        plt.suptitle(f"Augmentasi (season: {sample_image_season})")

    plt.tight_layout()
    plt.show()

In [ ]:
import os

# Get the list of season directories
season_directories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

# Count images in each season directory
season_counts = {}
for season_dir in season_directories:
    season_path = os.path.join(dataset_path, season_dir)
    image_files = [f for f in os.listdir(season_path) if os.path.isfile(os.path.join(season_path, f))]
    season_counts[season_dir] = len(image_files)

# Print the distribution
print("Distribusi gambar berdasarkan musim:")
for season, count in season_counts.items():
    print(f"{season}: {count} gambar")

In [ ]:
import matplotlib.pyplot as plt

# Get the season names and counts from the dictionary
seasons = list(season_counts.keys())
counts = list(season_counts.values())

# Create a bar plot
plt.figure(figsize=(10, 6))
plt.bar(seasons, counts, color=['skyblue', 'lightgreen', 'salmon', 'gold'])
plt.xlabel("Musim")
plt.ylabel("Jumlah Gambar")
plt.title("Distribusi Gambar Berdasarkan Musim")
plt.show()

In [ ]:
import os, glob, json, random, itertools, collections, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import top_k_accuracy_score
from sklearn.model_selection import KFold, train_test_split

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

if sample_image_path is None:
    print("Sample image belum tersedia. Jalankan cell pemilihan sample image.")
else:
    vis_img_height = globals().get("IMG_HEIGHT", 80)
    vis_img_width = globals().get("IMG_WIDTH", 60)

    # Visualisasi resizing
    img_original = tf.keras.utils.load_img(sample_image_path)
    img_resized = tf.keras.utils.load_img(
        sample_image_path,
        target_size=(vis_img_height, vis_img_width)
    )

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img_original)
    plt.title(f"Sebelum ({img_original.size[0]}x{img_original.size[1]})")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(img_resized)
    plt.title(f"Sesudah ({vis_img_width}x{vis_img_height})")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # Visualisasi normalisasi
    if "preprocess_input" not in globals():
        print("preprocess_input belum tersedia. Jalankan cell import MobileNetV2 preprocess_input.")
    else:
        img_array = tf.keras.utils.img_to_array(img_resized)
        img_batch = np.expand_dims(img_array, axis=0)
        preprocessed = preprocess_input(img_batch.copy())

        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1)
        plt.imshow(img_array.astype("uint8"))
        plt.title("Sebelum Normalisasi")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(preprocessed[0])
        plt.title("Sesudah Normalisasi")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

        print("=== Sebelum Normalisasi ===")
        print("Shape:", img_array.shape)
        print("Dtype:", img_array.dtype)
        print("Min value:", np.min(img_array))
        print("Max value:", np.max(img_array))

        print("\n=== Sesudah Normalisasi ===")
        print("Shape:", preprocessed.shape)
        print("Dtype:", preprocessed.dtype)
        print("Min value:", np.min(preprocessed))
        print("Max value:", np.max(preprocessed))

In [ ]:
TRAIN_DIR = "/kaggle/working"

In [ ]:
FOLD = 5

EPOCH = 100
LEARNING_RATE = 0.01
DROPOUT = 0.3

In [ ]:
def evaluate_with_report_cm(model, dataset, class_names, title_suffix=""):
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    # Classification Report
    print(f"\n Classification Report {title_suffix}")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            digits=4
        )
    )

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix {title_suffix}")
    plt.tight_layout()
    plt.show()

    return cm

In [ ]:
# Assuming TRAIN_DIR and TEST_DIR are already defined and contain the data
# Assuming num_classes is already defined (e.g., from previous cell XYvEd_61ZTnx)

kf = KFold(n_splits=FOLD, random_state=42, shuffle=True)

cvScores = []
i = 0 # Start fold index from 0

# Collect all image paths and labels ONLY from TRAIN_DIR
train_image_paths = []
train_labels = []

for class_folder in os.listdir(TRAIN_DIR):
    class_path = os.path.join(TRAIN_DIR, class_folder)
    if os.path.isdir(class_path):
        images = [os.path.join(class_path, img) for img in os.listdir(class_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        train_image_paths.extend(images)
        train_labels.extend([class_folder] * len(images))

# Create a mapping from class names to integer labels based on the original class names
# (Ensure this mapping is consistent with the num_classes used later)
class_names = sorted(list(set(train_labels))) # Get class names from train_dir
num_classes = len(class_names)
label_map = {name: idx for idx, name in enumerate(class_names)}
train_label_indices = [label_map[label] for label in train_labels]


# Convert to numpy arrays for KFold
train_image_paths = np.array(train_image_paths)
train_label_indices = np.array(train_label_indices)

# Define image size and batch size
IMG_WIDTH = 60
IMG_HEIGHT = 80
BATCH_SIZE = 32

# Define Data Augmentation (can be reused from previous cell jJRV6JjWc_ut)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")


for fold_index, (train_index, val_index) in enumerate(kf.split(train_image_paths)):
    print(f"Fold: {fold_index + 1} ==================================================================")

    # Split data for the current fold
    train_fold_paths, val_fold_paths = train_image_paths[train_index], train_image_paths[val_index]
    train_fold_labels, val_fold_labels = train_label_indices[train_index], train_label_indices[val_index]

    # Create new directories for the current fold
    fold_train_dir = f"/kaggle/working/fold/train_{fold_index}"
    fold_val_dir = f"/kaggle/working/fold/val_{fold_index}"

    # Clean up previous fold directories if they exist
    if os.path.exists(fold_train_dir):
        shutil.rmtree(fold_train_dir)
    if os.path.exists(fold_val_dir):
        shutil.rmtree(fold_val_dir)

    os.makedirs(fold_train_dir, exist_ok=True)
    os.makedirs(fold_val_dir, exist_ok=True)

    # Function to copy images to new fold directories
    def copy_images_to_fold_dir(image_paths, labels, dest_dir):
        for img_path, label_index in zip(image_paths, labels):
            class_name = class_names[label_index]
            dest_class_dir = os.path.join(dest_dir, class_name)
            os.makedirs(dest_class_dir, exist_ok=True)
            shutil.copy(img_path, os.path.join(dest_class_dir, os.path.basename(img_path)))

    # Copy images for the current fold
    copy_images_to_fold_dir(train_fold_paths, train_fold_labels, fold_train_dir)
    copy_images_to_fold_dir(val_fold_paths, val_fold_labels, fold_val_dir)


    # Create TensorFlow Datasets from the newly created fold directories
    def create_tf_dataset_from_fold_dir(directory, shuffle=True):
         dataset = tf.keras.utils.image_dataset_from_directory(
            directory, label_mode='categorical', image_size=(IMG_HEIGHT, IMG_WIDTH),
            batch_size=BATCH_SIZE, shuffle=shuffle, seed=None # No need for fixed seed here
         )
         # Ensure the dataset class names match the overall class names order
         # This is important for consistent label mapping during training/evaluation
        #  dataset = dataset.map(lambda x, y: (x, tf.gather(y, indices=tf.argsort(tf.constant(dataset.class_names)), axis=1)),
        #                        num_parallel_calls=tf.data.AUTOTUNE)

        #  dataset = dataset.cache().prefetch(tf.data.AUTOTUNE)
         return dataset


    train_ds_fold = create_tf_dataset_from_fold_dir(fold_train_dir, shuffle=True)
    val_ds_fold = create_tf_dataset_from_fold_dir(fold_val_dir, shuffle=False)

    train_ds_fold = train_ds_fold.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
    val_ds_fold = val_ds_fold.cache().prefetch(tf.data.AUTOTUNE)

    # Define the model (Ensure the base_model is reloaded or its weights are reset for each fold)
    base_model = MobileNetV2(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), include_top=False, weights="imagenet")
    base_model.trainable = False # Stage-1: freeze

    inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    x = data_augmentation(inputs)
    # Preprocess input specifically for ResNet50 (using its built-in function)
    x = layers.Lambda(preprocess_input, name="mobilenetv2_preproc")(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x) # dtype for mixed precision safety

    model = keras.Model(inputs, outputs, name=f"mobilenet_v2_season_fold_{fold_index}")

    model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
                  loss='categorical_crossentropy',
                  metrics = ['accuracy'])

    # Train the model
    print("Training model...")
    history_tl = model.fit(train_ds_fold, epochs=EPOCH, validation_data=val_ds_fold)

    # Evaluate on the test set for this fold
    print(f"\nEvaluating Fold {fold_index} on Test Set:")
    scores = model.evaluate(val_ds_fold, verbose=0) # Evaluate on validation set
    print(f"Test Loss: {scores[0]:.4f}")
    print(f"Test Accuracy: {scores[1]*100:.2f}%")
    cvScores.append(scores[1] * 100)

    evaluate_with_report_cm(
        model=model,
        dataset=val_ds_fold,
        class_names=class_names,
        title_suffix=f"(Transfer Learning - Fold {fold_index})"
    )

    history_df = pd.DataFrame(history_tl.history)

    history_df[['loss', 'val_loss']].plot()
    plt.title(f"Loss - Transfer Learning (Fold {fold_index})")
    plt.grid(True)
    plt.show()

    history_df[['accuracy', 'val_accuracy']].plot()
    plt.title(f"Accuracy - Transfer Learning (Fold {fold_index})")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.show()

    # Fine Tuning Model

    base_model.trainable = True
    for layer in base_model.layers[:-15]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Train the model
    print("Training model...")
    history_ft = model.fit(train_ds_fold, epochs=EPOCH, validation_data=val_ds_fold)

    # Evaluate on the test set for this fold
    print(f"\nEvaluating Fold {fold_index} on Test Set:")
    scores = model.evaluate(val_ds_fold, verbose=0) # Evaluate on validation set
    print(f"Test Loss: {scores[0]:.4f}")
    print(f"Test Accuracy: {scores[1]*100:.2f}%")
    cvScores.append(scores[1] * 100)

    evaluate_with_report_cm(
        model=model,
        dataset=val_ds_fold,
        class_names=class_names,
        title_suffix=f"(Fine Tuning - Fold {fold_index})"
    )

    history_df = pd.DataFrame(history_ft.history)

    history_df[['loss', 'val_loss']].plot()
    plt.title(f"Loss - Fine Tuning (Fold {fold_index})")
    plt.grid(True)
    plt.show()

    history_df[['accuracy', 'val_accuracy']].plot()
    plt.title(f"Accuracy - Fine Tuning (Fold {fold_index})")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.show()

    i += 1

# After the loop, print the cross-validation scores
print("\n==================================================================")
print("Cross-validation Accuracy Scores on Validation Set:")
for j, score in enumerate(cvScores):
    print(f"Fold {j}: {score:.2f}%")

print(f"\nMean CV Accuracy: {np.mean(cvScores):.2f}%")
print(f"Std Dev CV Accuracy: {np.std(cvScores):.2f}%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if len(cvScores) == 0:
    print("cvScores kosong. Jalankan proses K-Fold terlebih dahulu.")
else:
    if "FOLD" in globals() and len(cvScores) == FOLD * 2:
        tl_scores = cvScores[0::2]
        ft_scores = cvScores[1::2]
        folds = np.arange(1, len(tl_scores) + 1)

        plt.figure(figsize=(8, 4))
        plt.plot(folds, tl_scores, marker="o", label="Transfer Learning")
        plt.plot(folds, ft_scores, marker="o", label="Fine Tuning")
        plt.xticks(folds)
        plt.xlabel("Fold")
        plt.ylabel("Akurasi (%)")
        plt.title("Ringkasan Akurasi per Fold")
        plt.grid(True, axis="y", linestyle="--", alpha=0.5)
        plt.legend()
        plt.tight_layout()
        plt.show()

        print("=== Ringkasan Transfer Learning ===")
        print(f"Mean: {np.mean(tl_scores):.2f}%")
        print(f"Std: {np.std(tl_scores):.2f}%")

        print("\n=== Ringkasan Fine Tuning ===")
        print(f"Mean: {np.mean(ft_scores):.2f}%")
        print(f"Std: {np.std(ft_scores):.2f}%")
    else:
        folds = np.arange(1, len(cvScores) + 1)
        plt.figure(figsize=(8, 4))
        plt.plot(folds, cvScores, marker="o")
        plt.xticks(folds)
        plt.xlabel("Index")
        plt.ylabel("Akurasi (%)")
        plt.title("Ringkasan Akurasi")
        plt.grid(True, axis="y", linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()

        print("=== Ringkasan Akurasi ===")
        print(f"Mean: {np.mean(cvScores):.2f}%")
        print(f"Std: {np.std(cvScores):.2f}%")

In [ ]:
# Evaluate the model on the validation set using classification report and confusion matrix

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure val_ds_fold and class_names are available from the last fold of the K-Fold loop
# val_ds_fold is created in the K-Fold loop (cell 7BcZvym_YhhU) and should be available after the loop finishes
# class_names is also defined in cell 7BcZvym_YhhU

# 1) Kumpulkan y_true, y_prob, y_pred dari val_ds_fold dari lipatan terakhir
y_true_val, y_prob_val = [], []
# Iterate through the validation dataset of the last fold
for xb, yb in val_ds_fold:
    pb = model.predict(xb, verbose=0) # Use the model trained in the last fold
    y_prob_val.append(pb)
    y_true_val.append(np.argmax(yb.numpy(), axis=1))

y_prob_val = np.concatenate(y_prob_val, axis=0)   # shape: (N, num_classes)
y_true_val = np.concatenate(y_true_val, axis=0)   # shape: (N,)
y_pred_val = y_prob_val.argmax(axis=1)

# 2) Classification report (validation set)
print("\n[CLASSIFICATION REPORT] (Validation Set - Last Fold)")
print(classification_report(y_true_val, y_pred_val, target_names=class_names, digits=4))

# 3) Confusion matrix (validation set) - absolut & normalized
cm_val = confusion_matrix(y_true_val, y_pred_val)
cm_norm_val = cm_val.astype(float) / cm_val.sum(axis=1, keepdims=True)

plt.figure(figsize=(12,10))
sns.heatmap(cm_val, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix (Validation Set - Last Fold)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()

plt.figure(figsize=(12,10))
sns.heatmap(cm_norm_val, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Normalized Confusion Matrix (Validation Set - Last Fold)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()